In [1]:
import torch
from transformers import AutoTokenizer
from modeling.apertus.apertus_8b import ApertusModel

In [2]:
model = ApertusModel.from_pretrained(
    "swiss-ai/Apertus-v1.5-8B",
    device="cuda:4",
)
tokenizer = AutoTokenizer.from_pretrained("swiss-ai/Apertus-v1.5-8B")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] You are using a model of type `apertus1p5` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


In [5]:
special_tokens_dict

{'context': 0, 'question': 0, 'option_start': 0, 'option_end': 0, 'decide': 0}

In [4]:
special_tokens_dict = {
    "context": tokenizer.convert_tokens_to_ids("<|SPECIAL_100|>"),
    "question": tokenizer.convert_tokens_to_ids("<|SPECIAL_101|>"),
    "option_start": tokenizer.convert_tokens_to_ids("<|SPECIAL_102|>"),
    "option_end": tokenizer.convert_tokens_to_ids("<|SPECIAL_103|>"),
    "decide": tokenizer.convert_tokens_to_ids("<|SPECIAL_104|>"),
}


inv_special_tokens_dict = {v: k for k, v in special_tokens_dict.items()}

test_message = """<|SPECIAL_100|>I was charged twice.
<|SPECIAL_101|>Which department should handle this?
<|SPECIAL_102|>billing<|SPECIAL_103|>
<|SPECIAL_102|>shipping<|SPECIAL_103|>
<|SPECIAL_102|>returns<|SPECIAL_103|>
<|SPECIAL_104|>"""

token_ids = tokenizer(test_message, return_tensors="pt").to(next(model.parameters()).device)["input_ids"]

context_token_pos = (token_ids == special_tokens_dict["context"]).nonzero(as_tuple=True)[1].item()
question_token_pos = (token_ids == special_tokens_dict["question"]).nonzero(as_tuple=True)[1].item()
decide_token_pos = (token_ids == special_tokens_dict["decide"]).nonzero(as_tuple=True)[1].item()
option_start_token_positions = (token_ids == special_tokens_dict["option_start"]).nonzero(as_tuple=True)[1].tolist()
option_end_token_positions = (token_ids == special_tokens_dict["option_end"]).nonzero(as_tuple=True)[1].tolist()

print("=== Encoded Input ===")
print(token_ids)


RuntimeError: a Tensor with 0 elements cannot be converted to Scalar

In [ ]:
class PointerHead(torch.nn.Module):
    def __init__(self, hidden_dim=2560, pointer_dim=256):
        super().__init__()
        self.query = torch.nn.Linear(hidden_dim, pointer_dim)
        self.key = torch.nn.Linear(hidden_dim, pointer_dim)
        self.scale = pointer_dim ** -0.5

    def forward(self, decide, options):
        return (self.key(options) @ self.query(decide)) * self.scale

In [ ]:
with torch.inference_mode():
    # We assume BS 1
    hidden_state = model.partial_forward(token_ids)[0]
print(hidden_state.shape)

context_state = hidden_state[context_token_pos]
question_state = hidden_state[question_token_pos]
decide_state = hidden_state[decide_token_pos]
option_start_states = [hidden_state[pos] for pos in option_start_token_positions]
option_end_states = [hidden_state[pos] for pos in option_end_token_positions]
